In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage,SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o")

store = {}

def get_session_history(session_id:str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(llm,get_session_history)

config = {"configurable": {"session_id" :"abc"}}

response = with_message_history.invoke(
    [HumanMessage(content= "안녕 난 홍길동이야") ],
    config = config
)
print("AI:"+response.content)


AI:안녕하세요, 홍길동님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [5]:
config = {"configurable": {"session_id" :"abc"}}

response = with_message_history.invoke(
    [HumanMessage(content= "내이름이 뭐지?") ],
    config = config
)
print("AI:"+response.content)

AI:당신은 방금 전에 자신을 홍길동이라고 소개하셨어요. :)


In [6]:
config = {"configurable": {"session_id" :"42"}}

response = with_message_history.invoke(
    [HumanMessage(content= "내이름이 뭐지?") ],
    config = config
)
print("AI:"+response.content)

AI:죄송하지만, 당신의 이름을 알 수 있는 정보는 제공받지 못했습니다. 어떻게 도와드릴까요?


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage,SystemMessage

llm = ChatOpenAI(model="gpt-4o")

message = [
    SystemMessage(content= "너는 미녀와 야수에서 나오는 미녀임"),

    HumanMessage(content="안녕 저녁먹으래요")
]
llm.invoke(message)


AIMessage(content='안녕하세요! 어떤 맛있는 저녁을 드시게 될까요? 맛있는 식사가 되길 바랄게요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 32, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6084d5b3db', 'finish_reason': 'stop', 'logprobs': None}, id='run-c68eb0f7-bbee-48e1-8bed-99c9e2aa50b7-0', usage_metadata={'input_tokens': 32, 'output_tokens': 27, 'total_tokens': 59, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
result  = llm.invoke(message)
parser.invoke(result )

'안녕하세요! 오늘 저녁은 뭐 드실 예정이세요? 맛있는 거 드셨으면 좋겠어요!'

In [13]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a}의 역할이야 그케릭터에 맞게 사용자와 대화하라"
human_template = "안녕 저는 {character_b}입니다. 오늘 시간있다면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
("system", system_template),
("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
}  )

print(result)

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀의 역할이야 그케릭터에 맞게 사용자와 대화하라', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 저는 야수입니다. 오늘 시간있다면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [15]:
chain = prompt_template | llm | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
}  )


'안녕하세요, 야수씨! 물론 함께 저녁을 하는 건 정말 즐거운 일이겠어요. 오늘 저녁에 어떤 특별한 계획이 있으신가요? 혹시 함께 하고 싶은 활동이 있으면 말해주세요!'